# Demo: t-SNE — Iris & Digits Dataset

Notebook ini mendemonstrasikan teknik visualisasi **t-SNE** dari materi `t-SNE.md`,
diterapkan pada dataset **Iris** (150 sampel, 4 fitur) dan **Digits** (1797 sampel, 64 fitur).

Topik yang dibahas: preprocessing dengan StandardScaler, efek perplexity, efek random state,
interpretasi yang benar, dan visualisasi dataset berdimensi tinggi.

## 1. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_digits
import warnings
warnings.filterwarnings("ignore")

## 2. Dataset: Iris

Dataset Iris berisi 150 sampel bunga dari 3 species (*setosa*, *versicolor*, *virginica*),
masing-masing dengan 4 fitur numerik (panjang/lebar sepal dan petal).

Dengan hanya 4 fitur, kita sebenarnya masih bisa menggunakan pair plot.
Tapi dataset ini cocok untuk belajar t-SNE karena kecil dan hasilnya cepat.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

print(f"Shape: {X_iris.shape}")
print(f"Fitur: {iris.feature_names}")
print(f"Kelas: {list(iris.target_names)}")
print(f"Distribusi kelas: {np.bincount(y_iris)}")

## 3. Preprocessing: StandardScaler

t-SNE menghitung jarak Euclidean antar titik data. Jika fitur memiliki skala berbeda,
fitur dengan range besar akan mendominasi perhitungan jarak.

**StandardScaler** mengubah setiap fitur agar memiliki mean = 0 dan standard deviation = 1.

In [ ]:
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

print("Sebelum scaling:")
print(f"  Mean per fitur: {X_iris.mean(axis=0).round(2)}")
print(f"  Std per fitur:  {X_iris.std(axis=0).round(2)}")
print()
print("Sesudah scaling:")
print(f"  Mean per fitur: {X_iris_scaled.mean(axis=0).round(2)}")
print(f"  Std per fitur:  {X_iris_scaled.std(axis=0).round(2)}")

> **Penting**: Langkah StandardScaler ini **wajib** sebelum t-SNE. Tanpa scaling,
> fitur dengan range besar (misalnya petal length 1-7 cm) akan mendominasi
> dibanding fitur dengan range kecil (misalnya sepal width 2-4.5 cm).

## 4. t-SNE Dasar

Jalankan t-SNE dengan parameter default yang umum digunakan:
- `n_components=2` (reduksi ke 2D)
- `perplexity=30` (default sklearn)
- `random_state=42` (untuk reproduksibilitas)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_iris_tsne = tsne.fit_transform(X_iris_scaled)

print(f"Shape hasil t-SNE: {X_iris_tsne.shape}")
print(f"KL Divergence: {tsne.kl_divergence_:.4f}")
print(f"Iterasi berjalan: {tsne.n_iter_}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(iris.target_names):
    mask = y_iris == i
    ax.scatter(
        X_iris_tsne[mask, 0], X_iris_tsne[mask, 1],
        alpha=0.7, label=name.capitalize(), s=40,
        edgecolors="white", linewidth=0.5
    )
ax.legend(loc="best", title="Species")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE pada Dataset Iris (perplexity=30)")
plt.tight_layout()
plt.show()

Perhatikan bahwa **setosa** terpisah jelas dari dua species lainnya,
sementara **versicolor** dan **virginica** saling berdekatan.
Ini konsisten dengan fakta bahwa versicolor dan virginica memang
memiliki karakteristik morfologi yang lebih mirip.

> **Catatan**: Sumbu "t-SNE 1" dan "t-SNE 2" **tidak memiliki interpretasi**.
> Berbeda dengan PCA di mana PC1 menjelaskan arah variansi terbesar,
> sumbu t-SNE hanya merupakan hasil optimisasi dan nilainya tidak bermakna.

## 5. Efek Perplexity

Perplexity mengontrol seberapa banyak tetangga yang dipertimbangkan.
Mari bandingkan 4 nilai perplexity yang berbeda: 5, 15, 30, dan 50.

In [ ]:
perplexities = [5, 15, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, perp in zip(axes.flat, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_tsne = tsne.fit_transform(X_iris_scaled)

    for i, name in enumerate(iris.target_names):
        mask = y_iris == i
        ax.scatter(
            X_tsne[mask, 0], X_tsne[mask, 1],
            alpha=0.7, label=name.capitalize(), s=30,
            edgecolors="white", linewidth=0.3
        )
    ax.set_title(f"Perplexity = {perp}", fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

axes[0, 0].legend(loc="best", fontsize=8)
fig.suptitle("Efek Perplexity pada t-SNE (Dataset Iris)", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

**Observasi**:
- **Perplexity 5**: Klaster cenderung pecah menjadi sub-klaster kecil. Struktur sangat lokal.
- **Perplexity 15**: Klaster mulai terbentuk jelas, tapi mungkin ada fragmentasi.
- **Perplexity 30**: Keseimbangan yang baik. Klaster terlihat kohesif.
- **Perplexity 50**: Klaster lebih "menyebar", struktur global lebih terasa.

Untuk Iris (150 sampel), perplexity 15-30 biasanya memberikan hasil terbaik.
Perplexity harus selalu **lebih kecil dari jumlah data**.

> **Tips**: Dalam praktik, selalu coba minimal 3 nilai perplexity yang berbeda.
> Pola yang konsisten di berbagai perplexity lebih bisa dipercaya.

## 6. Efek Random State

t-SNE adalah algoritma **stokastik** — hasilnya bisa berbeda tiap kali dijalankan.
Mari bandingkan 4 random state yang berbeda dengan perplexity yang sama.

In [ ]:
seeds = [0, 42, 123, 456]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, seed in zip(axes.flat, seeds):
    tsne = TSNE(n_components=2, perplexity=30, random_state=seed)
    X_tsne = tsne.fit_transform(X_iris_scaled)

    for i, name in enumerate(iris.target_names):
        mask = y_iris == i
        ax.scatter(
            X_tsne[mask, 0], X_tsne[mask, 1],
            alpha=0.7, label=name.capitalize(), s=30,
            edgecolors="white", linewidth=0.3
        )
    ax.set_title(f"random_state = {seed}", fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

axes[0, 0].legend(loc="best", fontsize=8)
fig.suptitle("Efek Random State pada t-SNE (perplexity=30)", fontweight="bold", fontsize=13)
fig.tight_layout()
plt.show()

**Observasi**:
- Bentuk dan orientasi klaster berubah di setiap run.
- Tapi **pola dasarnya konsisten**: setosa selalu terpisah, versicolor-virginica selalu berdekatan.
- Posisi dan rotasi klaster memang bisa berubah — ini normal karena sumbu t-SNE tidak bermakna.

> **Awas**: Jangan pernah mengambil kesimpulan dari satu kali run saja.
> Selalu cek apakah pola yang terlihat konsisten di beberapa random state.
> Jika klaster muncul di satu seed tapi hilang di seed lain, itu kemungkinan artefak.

## 7. Interpretasi: Benar vs Salah

Bagian ini mendemonstrasikan kesalahan interpretasi paling umum pada t-SNE.

### Kesalahan 1: Ukuran Klaster

Ukuran (area) klaster di visualisasi t-SNE **tidak** merepresentasikan sebaran data yang sebenarnya.
t-SNE cenderung menyamakan kepadatan visual antar klaster.

### Kesalahan 2: Jarak Antar Klaster

Jarak antar klaster di t-SNE **tidak** bisa diinterpretasikan secara langsung.
Dua klaster yang berdekatan belum tentu lebih mirip dibanding yang berjauhan.

### Kesalahan 3: Satu Run Sudah Cukup

Seperti yang kita lihat di Section 6, hasil bisa bervariasi.
Selalu jalankan beberapa kali untuk memastikan robustness.

> **Awas**: Yang boleh diinterpretasikan dari t-SNE:
> - **Keberadaan klaster** — jika titik-titik konsisten mengelompok di beberapa run.
> - **Kedekatan dalam klaster** — titik-titik dalam satu klaster memang mirip.
>
> Yang **tidak boleh** diinterpretasikan:
> - Ukuran klaster (besar vs kecil)
> - Jarak antar klaster
> - Bentuk klaster (bulat vs memanjang)

In [ ]:
# Demonstrasi: jalankan t-SNE lalu hitung jarak antar centroid klaster
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_iris_scaled)

# Hitung centroid tiap klaster di ruang t-SNE
print("Jarak antar centroid di ruang t-SNE:")
centroids = []
for i, name in enumerate(iris.target_names):
    mask = y_iris == i
    centroid = X_tsne[mask].mean(axis=0)
    centroids.append(centroid)
    print(f"  {name}: ({centroid[0]:.1f}, {centroid[1]:.1f})")

centroids = np.array(centroids)
from scipy.spatial.distance import pdist, squareform
dist_tsne = squareform(pdist(centroids))

print("\nMatriks jarak (t-SNE):")
for i, name_i in enumerate(iris.target_names):
    for j, name_j in enumerate(iris.target_names):
        if j > i:
            print(f"  {name_i} - {name_j}: {dist_tsne[i, j]:.1f}")

# Bandingkan dengan jarak di ruang asli (scaled)
centroids_orig = []
for i in range(3):
    mask = y_iris == i
    centroids_orig.append(X_iris_scaled[mask].mean(axis=0))
centroids_orig = np.array(centroids_orig)
dist_orig = squareform(pdist(centroids_orig))

print("\nMatriks jarak (ruang asli, scaled):")
for i, name_i in enumerate(iris.target_names):
    for j, name_j in enumerate(iris.target_names):
        if j > i:
            print(f"  {name_i} - {name_j}: {dist_orig[i, j]:.2f}")

print("\n** Perhatikan: urutan jarak bisa berbeda antara t-SNE dan ruang asli!")
print("   Ini menunjukkan bahwa jarak antar klaster di t-SNE tidak bisa dipercaya.")

## 8. Dataset: Digits

Sekarang kita coba t-SNE pada dataset yang lebih menantang: **Digits**.
Dataset ini berisi 1797 gambar angka tulisan tangan (0-9), masing-masing berukuran 8x8 piksel = **64 fitur**.

Ini adalah contoh yang lebih realistis di mana t-SNE benar-benar bermanfaat —
tidak mungkin memvisualisasikan 64 dimensi dengan scatter plot biasa.

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

print(f"Shape: {X_dig.shape}")
print(f"Jumlah kelas: {len(np.unique(y_dig))}")
print(f"Distribusi kelas: {np.bincount(y_dig)}")

In [ ]:
# Tampilkan beberapa sampel gambar
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap="gray_r")
    ax.set_title(f"Label: {digits.target[i]}")
    ax.axis("off")
fig.suptitle("Sampel Gambar dari Dataset Digits (8x8 piksel)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Standarisasi lalu jalankan t-SNE
X_dig_scaled = StandardScaler().fit_transform(X_dig)

tsne_dig = TSNE(n_components=2, perplexity=30, random_state=42)
X_dig_tsne = tsne_dig.fit_transform(X_dig_scaled)

print(f"Shape hasil: {X_dig_tsne.shape}")
print(f"KL Divergence: {tsne_dig.kl_divergence_:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    X_dig_tsne[:, 0], X_dig_tsne[:, 1],
    c=y_dig, cmap=plt.cm.tab10,
    alpha=0.6, s=15, edgecolors="none"
)

cbar = fig.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label("Digit")
cbar.ax.set_yticklabels([str(d) for d in range(10)])

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE pada Dataset Digits (1797 sampel, 64 fitur)")
plt.tight_layout()
plt.show()

**Observasi**:
- t-SNE berhasil memisahkan 10 digit menjadi klaster yang cukup jelas.
- Beberapa digit (misalnya 4 dan 9) mungkin sedikit tumpang tindih — ini masuk akal karena
  bentuk tulisan tangan 4 dan 9 memang bisa mirip.
- Tanpa t-SNE, mustahil memvisualisasikan pemisahan ini dari 64 fitur.

> **Tips**: Nilai `kl_divergence_` yang lebih rendah menunjukkan embedding yang lebih baik.
> Jika nilainya sangat tinggi, coba naikkan `max_iter` atau sesuaikan `perplexity`.